# Rigveda Batched LLM Translator (Translation & Modernization)

This notebook is optimized to perform **highly accelerated GPU batched translation** of the pre-scraped Rigveda hymns. By scraping the website locally and processing translations in Colab, we bypass Cloudflare blocks and leverage Colab's free GPU compute.

### 📋 Execution Steps:
1. Run the local scraping script on your machine to generate `scraped_raw_hymns.json`:
   ```bash
   python local_scraper.py
   ```
2. Configure Colab GPU Runtime:
   - Go to **Runtime** > **Change runtime type** in the top menu.
   - Select **T4 GPU** and click **Save**.
3. Run Cell 1 to install standard translation libraries.
4. Run Cell 2 and upload `scraped_raw_hymns.json`.
5. Run the remaining cells in order to translate the hymns and download the consolidated modernized outputs in both **Text (`rigveda_modern.txt`)** and **JSON (`rigveda_modern.json`)** formats.

In [ ]:
# Install standard translation and LLM libraries
!pip install -q transformers accelerate bitsandbytes torch

## Upload Raw Scraped Data
Run this cell and upload the `scraped_raw_hymns.json` file generated by your local scraper.

In [ ]:
from google.colab import files
import os

if not os.path.exists("scraped_raw_hymns.json"):
    print("Please select and upload 'scraped_raw_hymns.json' from your machine:")
    uploaded = files.upload()
else:
    print("'scraped_raw_hymns.json' already detected in current directory. Ready to proceed!")

## Load Translation LLM (`Qwen/Qwen2.5-3B-Instruct`)

We load the model in 4-bit precision and configure the tokenizer for **left-padding** and **batched generation**.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-3B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Selected execution device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
# Configure left padding for batched inference
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if device == "cuda":
    print("GPU detected. Loading model in 4-bit precision...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    print("GPU not detected. Loading on CPU (WARNING: Inference will be extremely slow)...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="auto"
    )

model.config.pad_token_id = tokenizer.pad_token_id
print("Model loaded successfully!")

## Setup Batched Modernizer & Prompts

We define the system prompt and a function that processes a **batch of texts concurrently** through the GPU.

In [ ]:
from typing import List

system_prompt = (
    "You are an expert translator and Vedic scholar. Your task is to rewrite the given Rigvedic hymn from archaic/old English into clear, modern English.\n"
    "Follow these guidelines:\n"
    "1. Maintain the original meaning and spiritual essence. Do not add external interpretations.\n"
    "2. Use modern, natural English phrasing. Avoid archaic words like 'laud', 'thou', 'thy', 'hitherward', 'hotar', 'oblation', etc.\n"
    "3. Use standard Sanskrit/Hinglish terms for deities and key concepts instead of their English translations where appropriate. For example:\n"
    "   - Use 'Surya' instead of 'Sun' or 'sun god'.\n"
    "   - Use 'Agni' instead of 'Fire' or 'fire god'.\n"
    "   - Use 'Indra' instead of 'thunder god' or 'rain god'.\n"
    "   - Use 'Soma' for the sacred beverage.\n"
    "   - Use 'Yajna' or 'Havan' instead of 'sacrifice' or 'offering'.\n"
    "   - Use 'Deva' or 'Devas' instead of 'God' or 'Gods'.\n"
    "   - Use 'Mantra' or 'Shloka' instead of 'prayer' or 'song' or 'hymn'.\n"
    "   - Use 'Rishi' instead of 'seer' or 'priest'.\n"
    "4. Output ONLY the rewritten modern English translation of the verses. Do not include introductory notes, explanations, or chatty sentences."
)

def modernize_hymns_batch(hymn_texts: List[str]) -> List[str]:
    prompts = []
    for text in hymn_texts:
        if not text or not text.strip():
            prompts.append("")
            continue
            
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Original Hymn Content:\n{text}"}
        ]
        text_input = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        prompts.append(text_input)
        
    valid_prompts = [p for p in prompts if p]
    if not valid_prompts:
        return ["" for _ in hymn_texts]
        
    # Tokenize with left padding
    model_inputs = tokenizer(valid_prompts, return_tensors="pt", padding=True).to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate( 
            **model_inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
        
    responses = []
    valid_idx = 0
    for p in prompts:
        if not p:
            responses.append("")
        else:
            input_len = len(model_inputs.input_ids[valid_idx])
            hymn_generated_ids = generated_ids[valid_idx][input_len:]
            response = tokenizer.decode(hymn_generated_ids, skip_special_tokens=True)
            responses.append(response.strip())
            valid_idx += 1
            
    return responses

# Quick sanity check
sample_texts = [
    "1 I Laud Agni, the chosen Priest, God, minister of sacrifice, The hotar, lavishest of wealth."
]
print("Quick check: ", modernize_hymns_batch(sample_texts)[0])

## Run Translation Loop (with Resumable Checkpoints)

This cell flattens the scraped raw hymns, loads any previous translations from `rigveda_checkpoint.json` (to allow resumption), and processes the remaining hymns in batches of **16**.

In [ ]:
import os
import json
import time

# --- Configuration ---
BATCH_SIZE = 8           # 8 is highly stable, fast, and fits within Colab T4 GPU VRAM limits
CHECKPOINT_FILE = "rigveda_checkpoint.json"
RAW_DATA_FILE = "scraped_raw_hymns.json"
# ---------------------

if not os.path.exists(RAW_DATA_FILE):
    raise FileNotFoundError(f"Raw data file '{RAW_DATA_FILE}' not found! Upload the file in Cell 2.")

with open(RAW_DATA_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Flatten hymns list
all_hymns = []
for book in raw_data["books"]:
    for hymn in book["hymns"]:
        all_hymns.append({
            "book_title": book["title"],
            "hymn_title": hymn["title"],
            "url": hymn["url"],
            "content": hymn["content"]
        })

total_hymns = len(all_hymns)
print(f"Total hymns loaded for translation: {total_hymns}")

# Load checkpoint
translated_hymns = []
start_idx = 0

if os.path.exists(CHECKPOINT_FILE):
    try:
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            translated_hymns = json.load(f)
        start_idx = len(translated_hymns)
        print(f"Checkpoint loaded! Already translated {start_idx}/{total_hymns} hymns. Resuming...")
    except Exception as e:
        print(f"Could not load checkpoint ({e}). Starting from scratch.")
        translated_hymns = []

# Translation loop
if start_idx < total_hymns:
    if model.device.type == "cpu":
        print("\n[WARNING]: Model is loaded on CPU. Batched generation will be extremely slow.")
        print("Please change the Colab runtime to T4 GPU (Runtime -> Change runtime type) for optimal performance.\n")
        
    start_time = time.time()
    for i in range(start_idx, total_hymns, BATCH_SIZE):
        batch = all_hymns[i : i + BATCH_SIZE]
        batch_contents = [hymn["content"] for hymn in batch]
        
        print(f"Translating batch {i // BATCH_SIZE + 1}... (Hymns {i+1} to {min(i + BATCH_SIZE, total_hymns)})", end="", flush=True)
        batch_start = time.time()
        
        try:
            batch_translations = modernize_hymns_batch(batch_contents)
        except Exception as e:
            print(f"\nError translating batch starting at index {i}: {e}. Retrying individually...")
            batch_translations = []
            for hymn in batch:
                try:
                    batch_translations.append(modernize_hymns_batch([hymn["content"]])[0])
                except Exception as e_ind:
                    print(f"Failed individual translation for {hymn['hymn_title']}: {e_ind}")
                    batch_translations.append(f"[Translation Error]: {hymn['content']}")
                    
        for j, hymn in enumerate(batch):
            translated_hymns.append({
                "book_title": hymn["book_title"],
                "hymn_title": hymn["hymn_title"],
                "url": hymn["url"],
                "original_content": hymn["content"],
                "rewritten_content": batch_translations[j]
            })
            
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            json.dump(translated_hymns, f, ensure_ascii=False, indent=2)
            
        elapsed = time.time() - start_time
        batch_elapsed = time.time() - batch_start
        processed = len(translated_hymns)
        speed = processed / elapsed if elapsed > 0 else 0
        eta_seconds = (total_hymns - processed) / speed if speed > 0 else 0
        eta_mins = eta_seconds / 60
        
        print(f" -> Done in {batch_elapsed:.1f}s | Progress: {processed}/{total_hymns} ({(processed/total_hymns)*100:.1f}%) | ETA: {eta_mins:.1f} mins", flush=True)
        
    print(f"\nTranslation complete! Modernized {len(translated_hymns)} hymns in {elapsed/60:.1f} minutes.")
else:
    print(f"All {total_hymns} loaded hymns are already translated!")

## Phase 3: Consolidation & Browser Download (TXT + JSON)

This cell formats the translations into both a clean text file `rigveda_modern.txt` and a structured JSON file `rigveda_modern.json` (preserving the exact input structure from `scraped_raw_hymns.json`), and downloads both automatically.

In [ ]:
import json
from collections import defaultdict

CHECKPOINT_FILE = "rigveda_checkpoint.json"
RAW_DATA_FILE = "scraped_raw_hymns.json"
OUTPUT_TEXT_FILE = "rigveda_modern.txt"
OUTPUT_JSON_FILE = "rigveda_modern.json"

# 1. Load results and raw structure
with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    results = json.load(f)

with open(RAW_DATA_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
    
# 2. Create the output TXT file
books_dict = defaultdict(list)
for item in results:
    books_dict[item["book_title"]].append(item)
    
output_lines = [
    "RIG VEDA - MODERN ENGLISH & HINGLISH REWRITE",
    "=" * 50,
    ""
]

for book_title, hymns in books_dict.items():
    output_lines.append("=" * 50)
    output_lines.append(book_title.upper())
    output_lines.append("=" * 50)
    output_lines.append("")
    
    for hymn in hymns:
        output_lines.append("-" * 50)
        output_lines.append(f"{hymn['hymn_title']}")
        output_lines.append(f"URL: {hymn['url']}")
        output_lines.append("-" * 50)
        output_lines.append(hymn["rewritten_content"])
        output_lines.append("")
        output_lines.append("")
        
with open(OUTPUT_TEXT_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines))
print(f"TXT output successfully saved to {OUTPUT_TEXT_FILE}.")

# 3. Create the output JSON file in identical format as scraped_raw_hymns.json
translation_map = {}
for item in results:
    key = (item["book_title"], item["hymn_title"])
    translation_map[key] = item["rewritten_content"]

modern_json_data = {
    "book_title": raw_data.get("book_title", "Rig Veda"),
    "index_url": raw_data.get("index_url", "https://sacred-texts.com/hin/rigveda/index.htm"),
    "books": []
}

for book in raw_data["books"]:
    modern_book = {
        "title": book["title"],
        "url": book["url"],
        "hymns": []
    }
    for hymn in book["hymns"]:
        key = (book["title"], hymn["title"])
        rewritten = translation_map.get(key, "")
        modern_book["hymns"].append({
            "title": hymn["title"],
            "url": hymn["url"],
            "content": rewritten
        })
    modern_json_data["books"].append(modern_book)

with open(OUTPUT_JSON_FILE, "w", encoding="utf-8") as f:
    json.dump(modern_json_data, f, ensure_ascii=False, indent=2)
print(f"JSON output successfully saved to {OUTPUT_JSON_FILE}.")

# 4. Trigger downloads in Google Colab
try:
    from google.colab import files
    print("Starting browser download for TXT file...")
    files.download(OUTPUT_TEXT_FILE)
    time.sleep(1.0) # Small pause to prevent browser blocking multiple downloads
    print("Starting browser download for JSON file...")
    files.download(OUTPUT_JSON_FILE)
except ImportError:
    print("[Notice] google.colab is not available. Download link skipped.")
    print(f"You can find files locally at: {OUTPUT_TEXT_FILE} and {OUTPUT_JSON_FILE}")